# Reduce Images Weight for Games Gallery

This notebook optimizes images in the games gallery by reducing their file size while maintaining quality.

**Strategy:**
- Process all images in `static/img/games-gallery/`
- Skip images smaller than 500KB
- Optimize larger images using Pillow with high quality settings
- Track statistics on size reduction

In [ ]:
# Import required libraries
from pathlib import Path
 
from PIL import Image

print("Libraries imported successfully!")

In [ ]:
# Configuration
BASE_DIR = Path(r"C:\Users\danyp\Documents\Programmazione\Github Repos\MindMash-private\static\img\games-gallery") # noqa
MIN_SIZE_KB = 500  # Skip images smaller than this
MIN_SAVINGS_KB = 50  # Minimum savings required to keep the optimization
QUALITY = 90  # JPEG quality (1-100, higher = better quality but larger file)
OPTIMIZE = True  # Enable PIL optimization

# Supported image formats
IMAGE_EXTENSIONS = {'.png', '.jpg', '.jpeg', '.webp', '.gif'}

print(f"Base directory: {BASE_DIR}")
print(f"Will process images larger than {MIN_SIZE_KB}KB")
print(f"Minimum savings required: {MIN_SAVINGS_KB}KB")
print(f"Target quality: {QUALITY}")

In [ ]:
def get_file_size_kb(file_path: Path) -> float:
    """Get file size in KB."""
    return file_path.stat().st_size / 1024


def optimize_image(image_path: Path, quality: int = 85, min_savings_kb: float = 50) -> tuple[float, float, Path | None]:
    """
    Optimize an image file.

    Args:
        image_path: Path to the image file
        quality: JPEG quality setting (1-100)
        min_savings_kb: Minimum KB to save for optimization to be applied

    Returns
    -------
        Tuple of (original_size_kb, new_size_kb, new_path_if_converted)
        new_path_if_converted is None if no format change, otherwise the new path
        If optimization doesn't save enough space, returns original sizes
    """
    import shutil # noqa
    import tempfile # noqa

    original_size = get_file_size_kb(image_path)

    # Create a backup of the original file
    with tempfile.NamedTemporaryFile(delete=False, suffix=image_path.suffix) as tmp:
        backup_path = Path(tmp.name)
    shutil.copy2(image_path, backup_path)

    try:
        # Open the image
        img = Image.open(image_path)

        # Convert RGBA to RGB if necessary (for JPEG compatibility)
        if img.mode in ('RGBA', 'LA', 'P'):
            # Create a white background
            background = Image.new('RGB', img.size, (255, 255, 255))
            if img.mode == 'P':
                img = img.convert('RGBA')
            background.paste(img, mask=img.split()[-1] if img.mode in ('RGBA', 'LA') else None)
            img = background

        # Determine save format
        extension = image_path.suffix.lower()

        # Save optimized image
        save_kwargs = {'optimize': OPTIMIZE}
        converted_path = None
        final_path = image_path

        if extension in ['.jpg', '.jpeg']:
            save_kwargs['quality'] = quality
            save_kwargs['format'] = 'JPEG'
            img.save(image_path, **save_kwargs)
        elif extension == '.png':
            # For PNG, we'll convert to JPEG if it's not transparent
            # This usually results in better compression
            save_kwargs['quality'] = quality
            save_kwargs['format'] = 'JPEG'
            # Change extension to .jpg
            new_path = image_path.with_suffix('.jpg')
            img.save(new_path, **save_kwargs)
            final_path = new_path
            converted_path = new_path
        elif extension == '.webp':
            save_kwargs['quality'] = quality
            save_kwargs['format'] = 'WEBP'
            img.save(image_path, **save_kwargs)
        else:
            # For other formats, just optimize
            img.save(image_path, **save_kwargs)

        new_size = get_file_size_kb(final_path)
        savings = original_size - new_size

        # Check if savings meet minimum threshold
        if savings < min_savings_kb:
            # Revert the optimization - not worth it
            shutil.copy2(backup_path, image_path)

            # If we created a converted file (PNG -> JPEG), remove it
            if converted_path and converted_path.exists() and converted_path != image_path:
                converted_path.unlink()

            # Clean up backup
            backup_path.unlink()

            # Return original size for both (no change)
            return original_size, original_size, None

        # Optimization successful and worthwhile
        # If PNG was converted to JPEG, delete the original PNG
        if converted_path and converted_path != image_path and image_path.exists():
            image_path.unlink()  # Delete the original PNG

        # Clean up backup
        backup_path.unlink()

        return original_size, new_size, converted_path

    except Exception as e:
        # Restore from backup on error
        shutil.copy2(backup_path, image_path)
        backup_path.unlink()
        raise e


def format_size(size_kb: float) -> str:
    """Format file size in human-readable format."""
    if size_kb < 1024: # noqa
        return f"{size_kb:.2f} KB"
    return f"{size_kb/1024:.2f} MB"


print("Helper functions defined successfully!")

In [ ]:
# Discover all image files
all_images = []

for year_dir in sorted(BASE_DIR.iterdir()):
    if not year_dir.is_dir():
        continue

    # Get all image files in the year directory (posters)
    for file in year_dir.iterdir():
        if file.is_file() and file.suffix.lower() in IMAGE_EXTENSIONS:
            all_images.append(file)

        # Get images in subdirectories (screenshots)
        elif file.is_dir():
            for screenshot in file.rglob("*"):
                if screenshot.is_file() and screenshot.suffix.lower() in IMAGE_EXTENSIONS:
                    all_images.append(screenshot) # noqa

print(f"Found {len(all_images)} total images")

# Filter images that need processing (> MIN_SIZE_KB)
images_to_process = []
total_size_before = 0

for img_path in all_images:
    size_kb = get_file_size_kb(img_path)
    total_size_before += size_kb

    if size_kb > MIN_SIZE_KB:
        images_to_process.append(img_path)

print(f"\nImages to process: {len(images_to_process)} (larger than {MIN_SIZE_KB}KB)")
print(f"Images to skip: {len(all_images) - len(images_to_process)} (already optimized)")
print(f"\nTotal size before optimization: {format_size(total_size_before)}")

In [ ]:
# Process images
total_saved = 0
processed_count = 0
skipped_count = 0  # Images that didn't meet minimum savings
failed_count = 0
failed_images = []
converted_images = []  # Track PNG -> JPEG conversions
skipped_images = []  # Track images skipped due to insufficient savings

print("Starting image optimization...\n")
print("=" * 80)

for i, img_path in enumerate(images_to_process, 1):
    try:
        original_size, new_size, converted_path = optimize_image(
            img_path, quality=QUALITY, min_savings_kb=MIN_SAVINGS_KB
        )
        saved = original_size - new_size

        if saved > 0:
            # Optimization was applied
            total_saved += saved
            processed_count += 1

            # Track conversions
            if converted_path:
                converted_images.append((img_path, converted_path))

            # Show progress every 10 images or for significant savings
            if i % 10 == 0 or saved > 500: # noqa
                reduction_percent = (saved / original_size * 100) if original_size > 0 else 0
                relative_path = converted_path.relative_to(BASE_DIR) if converted_path else img_path.relative_to(BASE_DIR) # noqa
                print(f"[{i}/{len(images_to_process)}] {relative_path}")
                print(f"   {format_size(original_size)} → {format_size(new_size)} "
                      f"(saved {format_size(saved)}, {reduction_percent:.1f}% reduction)")
                if converted_path:
                    print(f"   🔄 Converted: {img_path.suffix} → {converted_path.suffix}")
        else:
            # Optimization was skipped (didn't meet minimum savings)
            skipped_count += 1
            skipped_images.append(img_path)
            if i % 10 == 0:
                print(f"[{i}/{len(images_to_process)}] ⏭️  Skipped: {img_path.relative_to(BASE_DIR)} (< {MIN_SAVINGS_KB}KB savings)")

    except Exception as e: # noqa
        failed_count += 1
        failed_images.append((img_path, str(e)))
        print(f"[{i}/{len(images_to_process)}] ❌ FAILED: {img_path.relative_to(BASE_DIR)}")
        print(f"   Error: {e!s}")

print("=" * 80)
print("\n✅ Optimization complete!")
print(f"\nProcessed: {processed_count}/{len(images_to_process)} images")
print(f"Skipped (< {MIN_SAVINGS_KB}KB savings): {skipped_count} images")
print(f"Converted PNG→JPEG: {len(converted_images)} images")
print(f"Failed: {failed_count} images")
print(f"Total space saved: {format_size(total_saved)}")

if converted_images:
    print("\n🔄 Converted images (PNG deleted, JPEG created):")
    for old_path, new_path in converted_images[:10]:  # Show first 10
        print(f"  ✓ {old_path.name} → {new_path.name}")
    if len(converted_images) > 10: # noqa
        print(f"  ... and {len(converted_images) - 10} more")

if skipped_images:
    print(f"\n⏭️  Skipped {len(skipped_images)} images (savings < {MIN_SAVINGS_KB}KB):")
    for img_path in skipped_images[:5]:
        print(f"  - {img_path.relative_to(BASE_DIR)}")
    if len(skipped_images) > 5:
        print(f"  ... and {len(skipped_images) - 5} more")

if failed_images:
    print("\n⚠️ Failed images:")
    for img_path, error in failed_images:
        print(f"  - {img_path.relative_to(BASE_DIR)}: {error}")

In [ ]:
# Final statistics
total_size_after = sum(get_file_size_kb(img) for img in all_images if img.exists())
total_reduction = total_size_before - total_size_after
reduction_percent = (total_reduction / total_size_before * 100) if total_size_before > 0 else 0

print("📊 FINAL STATISTICS")
print("=" * 80)
print(f"Total images in gallery: {len(all_images)}")
print(f"Images optimized: {processed_count}")
print(f"Images skipped (< {MIN_SIZE_KB}KB): {len(all_images) - len(images_to_process)}")
print(f"\nSize before: {format_size(total_size_before)}")
print(f"Size after:  {format_size(total_size_after)}")
print(f"\n💾 Total saved: {format_size(total_reduction)} ({reduction_percent:.1f}% reduction)")
print("=" * 80)

In [ ]:
# Verification: Check for orphaned PNG files
print("🔍 VERIFICATION CHECK")
print("=" * 80)

# Find all remaining PNG files in screenshot directories (not root level posters)
remaining_pngs = []
for year_dir in sorted(BASE_DIR.iterdir()):
    if not year_dir.is_dir():
        continue

    for subdir in year_dir.iterdir():
        if subdir.is_dir():
            for png_file in subdir.glob("*.png"):
                # Check if corresponding JPEG exists
                jpg_equivalent = png_file.with_suffix('.jpg')
                if jpg_equivalent.exists():
                    remaining_pngs.append((png_file, jpg_equivalent))

if remaining_pngs:
    print(f"⚠️ Found {len(remaining_pngs)} PNG files that should have been converted:")
    for png, jpg in remaining_pngs[:5]:
        print(f"  - {png.relative_to(BASE_DIR)} (JPEG exists: {jpg.name})")
    if len(remaining_pngs) > 5: # noqa
        print(f"  ... and {len(remaining_pngs) - 5} more")
    print("\n💡 These PNGs can be safely deleted manually if needed.")
else:
    print("✅ No orphaned PNG files found - all conversions successful!")

print("=" * 80)

## 📝 Notes

### What this notebook does:
1. **Scans** all images in `static/img/games-gallery/` for all years
2. **Identifies** images larger than 500KB
3. **Optimizes** those images using Pillow with quality=85
4. **Converts** PNGs to JPEGs for better compression (automatically deletes original PNG)
5. **Reports** total space saved and lists all conversions
6. **Verifies** no orphaned PNG files remain after conversion

### Quality Settings:
- **Quality 90**: High quality balance between file size and visual quality
- **Optimize flag**: Enabled for additional compression
- **PNG → JPEG conversion**: Automatic for better compression (original PNG deleted)

### Important:
- ⚠️ This process is **irreversible** - make a backup if needed!
- ✅ Skips images already < 500KB (considered already optimized)
- ✅ Shows progress every 10 images processed
- ✅ **PNG files are automatically deleted** after conversion to JPEG
- ✅ JSON files use `"images": true` for auto-discovery, so no updates needed

### What happens to PNG files:
1. PNG is converted to JPEG with high quality
2. Original PNG is **deleted** automatically
3. Only the new JPEG remains
4. Webpack auto-discovery picks up the new JPEG files automatically

### To re-run with different settings:
Modify the `QUALITY` parameter in the configuration cell (values: 1-100, default 90)